# 8. BREAST CANCER WISCONSIN - Preprocesamiento
**Tipo:** CLASIFICACIÓN BINARIA (tumor benigno/maligno)
**Variable objetivo:** Class (2=Benigno→0, 4=Maligno→1)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# CARGAR DATOS
# ============================================================
columnas = ['id', 'clump_thickness', 'uniformity_size', 'uniformity_shape',
            'marginal_adhesion', 'epithelial_size', 'bare_nuclei',
            'bland_chromatin', 'normal_nucleoli', 'mitoses', 'class']

df = pd.read_csv('breast-cancer-wisconsin.data', names=columnas, na_values='?')
print(f"Dimensiones: {df.shape}")
df.head()

In [ ]:
# ============================================================
# LIMPIEZA DE NULOS (están como "?")
# ============================================================
print("Nulos por columna:")
print(df.isnull().sum())

# La columna 'bare_nuclei' tiene nulos
# Opción 1: Eliminar filas con nulos (solo 16 filas)
df = df.dropna()
print(f"\nFilas después de limpieza: {len(df)}")

In [ ]:
# ============================================================
# PREPARAR VARIABLE OBJETIVO
# ============================================================
# 2 = Benigno → 0
# 4 = Maligno → 1
df['class'] = df['class'].map({2: 0, 4: 1})
y = df['class'].values

print(f"Balance de clases: {np.bincount(y)}")
print(f"Proporción malignos: {y.mean():.2%}")

In [ ]:
# ============================================================
# PREPARAR CARACTERÍSTICAS
# ============================================================
# Eliminar 'id' (no es característica médica) y 'class' (objetivo)
X_df = df.drop(columns=['id', 'class'])

# Asegurar que todas las columnas son numéricas
X_df = X_df.astype(float)

print(f"Características: {list(X_df.columns)}")
print(f"Todas en escala 1-10, muy amigable para ML")

In [ ]:
# ============================================================
# DIVISIÓN DE DATOS
# ============================================================
X = X_df.to_numpy()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.333, random_state=42)

print(f"Train: {X_train.shape[0]}")
print(f"Val: {X_val.shape[0]}")
print(f"Test: {X_test.shape[0]}")

In [ ]:
# ============================================================
# NORMALIZACIÓN
# ============================================================
# Aunque todo está en escala 1-10, normalizar ayuda al modelo
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("✅ Datos listos para CLASIFICACIÓN BINARIA")
print(f"Características: {X_train.shape[1]}")

## MÉTODO TRADICIONAL

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, recall_score

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall (detectar malignos): {recall_score(y_test, y_pred):.4f}")
print("\n⚠️ En cáncer, el RECALL es crítico (no queremos falsos negativos)")
print(classification_report(y_test, y_pred))

## CON PYTORCH

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).reshape(-1, 1)
X_test_t = torch.FloatTensor(X_test)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

In [ ]:
# Modelo simple para dataset pequeño
class DetectorCancer(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.red(x)

modelo = DetectorCancer(X_train_t.shape[1])
criterio = nn.BCELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)
print(modelo)
print(f"\nInput: {X_train_t.shape[1]} características")